# 🧠 Notebook 8 — Semantic Intelligence Engine
## Fake Internship & Job Scam Detection System

---

### 📌 Project Overview

| Detail | Info |
|--------|------|
| **Project** | Fake Internship & Job Scam Detection System |
| **Notebook** | 08 — Semantic Intelligence Engine |
| **Depends On** | Notebook 5 (Detection Engine), Notebook 6 (URL Analysis), Notebook 7 (Community Search) |
| **Purpose** | Vector embedding–based semantic search and pattern discovery |
| **Model** | `all-MiniLM-L6-v2` via SentenceTransformers |

---

### 🎯 Why Semantic Search?

Notebook 7 handles **keyword + fuzzy** matching. Notebook 8 goes deeper:
it understands *meaning*, not just characters.

| Query | Keyword Match | Semantic Match |
|-------|---------------|----------------|
| `ABC Consulting` | ABC Consulting, ABC Consultancy | + ABC Career Services, ABC Recruiters, ABC Placement Agency |
| `registration fee` | registration fee | + security deposit, joining contribution, processing amount, investment required |

---

### 📂 Notebook Flow

```
STEP 01 → Import Libraries
STEP 02 → Load Data
STEP 03 → Build Semantic Corpus
STEP 04 → Load Embedding Model
STEP 05 → Generate Embeddings
STEP 06 → Save Embeddings
STEP 07 → Semantic Search Function
STEP 08 → Similar Company Discovery
STEP 09 → Scam Pattern Discovery
STEP 10 → Community Intelligence Layer
STEP 11 → Risk Aggregation
STEP 12 → Frontend Search Response
STEP 13 → Test Cases
STEP 14 → Export Results
```

---
## STEP 01 — Import Libraries

In [1]:
# ============================================================
#  STEP 01 — IMPORT LIBRARIES
#  Semantic Intelligence Engine | Notebook 8
# ============================================================

# Core data processing
import pandas as pd
import numpy as np

# Semantic embedding model
from sentence_transformers import SentenceTransformer

# Cosine similarity for vector comparison
from sklearn.metrics.pairwise import cosine_similarity

# Model / embedding persistence
import joblib

# Utilities
import json
import os
import re
import warnings
from datetime import datetime
from collections import Counter

# Display settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 130)

print(" All libraries imported successfully.")
print(f"   pandas              : {pd.__version__}")
print(f"   numpy               : {np.__version__}")
print(f"   sentence_transformers: (SentenceTransformer ready)")
print(f"   sklearn             : (cosine_similarity ready)")
print(f"   joblib              : {joblib.__version__}")

 All libraries imported successfully.
   pandas              : 2.2.2
   numpy               : 1.26.4
   sentence_transformers: (SentenceTransformer ready)
   sklearn             : (cosine_similarity ready)
   joblib              : 1.4.2


---
## STEP 02 — Load Data

Load four data sources:
- `community_posts.csv`
- `community_comments.csv`
- `scam_reports.csv`
- `verified_scam_reports.csv`

Synthetic data is generated automatically when files are absent.

In [2]:
# ============================================================
#  STEP 02 — LOAD DATA
# ============================================================

DATA_DIR   = "../data/"
MODELS_DIR = "../models/"
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# ----------------------------------------------------------
#  Synthetic data generators
# ----------------------------------------------------------

def gen_posts():
    rows = [
        {"post_id":"P001","company_name":"ABC Consulting","title":"Beware of ABC Consulting fake internship","content":"They asked me to pay a registration fee of Rs 2000 to secure an internship. Classic red flag.","author":"user_rahul","date":"2024-01-15","likes":45,"number_of_comments":12,"tags":"scam,internship,registration fee"},
        {"post_id":"P002","company_name":"ABC Consultancy","title":"ABC Consultancy demanded security deposit","content":"ABC Consultancy sent offer letter but asked Rs 5000 security deposit before joining date.","author":"user_priya","date":"2024-01-18","likes":38,"number_of_comments":9,"tags":"scam,security deposit"},
        {"post_id":"P003","company_name":"TCS","title":"Fake TCS offer via WhatsApp asking fee","content":"Someone impersonating TCS HR on WhatsApp requesting Rs 5000 processing fee for onboarding.","author":"user_amit","date":"2024-01-20","likes":102,"number_of_comments":34,"tags":"TCS,whatsapp,fake offer"},
        {"post_id":"P004","company_name":"TCS","title":"TCS never charges any fee — official note","content":"TCS official site confirms they never ask candidates for money at any stage of hiring.","author":"user_neha","date":"2024-01-22","likes":87,"number_of_comments":21,"tags":"TCS,official"},
        {"post_id":"P005","company_name":"Infosys","title":"Fake Infosys internship on LinkedIn","content":"Received a LinkedIn message from fake Infosys HR asking Rs 3000 for training material kit.","author":"user_kiran","date":"2024-02-01","likes":67,"number_of_comments":18,"tags":"Infosys,LinkedIn,training fee"},
        {"post_id":"P006","company_name":"ABC Career Consulting","title":"ABC Career Consulting is a placement scam","content":"ABC Career Consulting charged Rs 10000 for guaranteed placement assistance. No job was ever given.","author":"user_suresh","date":"2024-02-10","likes":74,"number_of_comments":23,"tags":"scam,placement,money"},
        {"post_id":"P007","company_name":"ABC Recruiters","title":"ABC Recruiters asked joining contribution","content":"ABC Recruiters called it a joining contribution but it was Rs 3500 to be paid before offer release.","author":"user_divya","date":"2024-02-14","likes":51,"number_of_comments":15,"tags":"scam,joining fee"},
        {"post_id":"P008","company_name":"ABC Placement Agency","title":"ABC Placement Agency investment required scam","content":"ABC Placement Agency told me to invest Rs 2000 as a refundable deposit that was never returned.","author":"user_ravi","date":"2024-02-17","likes":44,"number_of_comments":11,"tags":"scam,investment,fake"},
        {"post_id":"P009","company_name":"Global Placement Hub","title":"Global Placement Hub stole registration fee","content":"Paid Rs 8000 registration fee to Global Placement Hub. Company disappeared after collecting money.","author":"user_raj","date":"2024-02-15","likes":33,"number_of_comments":7,"tags":"scam,placement,registration fee"},
        {"post_id":"P010","company_name":"WhatsApp Jobs","title":"Never trust job offers received on WhatsApp","content":"All job offers through WhatsApp groups asking for money are fraud. Never pay anyone over WhatsApp.","author":"user_anita","date":"2024-02-12","likes":120,"number_of_comments":45,"tags":"whatsapp,fake,awareness"},
        {"post_id":"P011","company_name":"TCS iON","title":"Fake TCS iON email from Gmail ID","content":"Received fake TCS iON offer from tcs-ion-hr@gmail.com. TCS uses @tcs.com domain exclusively.","author":"user_vikram","date":"2024-02-20","likes":89,"number_of_comments":29,"tags":"TCS,email,fake"},
        {"post_id":"P012","company_name":"ABC Career Services","title":"ABC Career Services processing amount fraud","content":"ABC Career Services demanded a processing amount of Rs 1500 before sharing company contact details.","author":"user_meena","date":"2024-02-25","likes":39,"number_of_comments":10,"tags":"scam,processing fee"},
        {"post_id":"P013","company_name":"Telegram Jobs","title":"Telegram job groups are often fake recruitment traps","content":"Beware of job groups on Telegram where unknown recruiters ask for document submission fees.","author":"user_aryan","date":"2024-03-01","likes":77,"number_of_comments":22,"tags":"telegram,fake,scam"},
        {"post_id":"P014","company_name":"Infosys BPM","title":"Infosys BPM security deposit demanded before joining","content":"Fake Infosys BPM HR asked Rs 4000 security deposit to be paid via GPay before releasing offer letter.","author":"user_maya","date":"2024-03-05","likes":56,"number_of_comments":17,"tags":"Infosys,BPM,security deposit"},
    ]
    return pd.DataFrame(rows)

def gen_comments():
    rows = [
        {"comment_id":"C001","post_id":"P001","company_name":"ABC Consulting","comment":"I got the same call from ABC Consulting. They even sent a fake offer letter. Pure scam!","author":"user_arun","date":"2024-01-16","upvotes":15},
        {"comment_id":"C002","post_id":"P003","company_name":"TCS","comment":"TCS official HR never contacts on personal WhatsApp. Always verify from tcs.com careers page.","author":"user_geeta","date":"2024-01-21","upvotes":35},
        {"comment_id":"C003","post_id":"P006","company_name":"ABC Career Consulting","comment":"ABC Career Consulting and ABC Recruiters are operated by the same group. Different brand, same fraud.","author":"user_tanya","date":"2024-02-11","upvotes":33},
        {"comment_id":"C004","post_id":"P007","company_name":"ABC Recruiters","comment":"Any company calling a fee a joining contribution or investment is asking for a bribe. Walk away.","author":"user_mohit","date":"2024-02-15","upvotes":48},
        {"comment_id":"C005","post_id":"P010","company_name":"WhatsApp Jobs","comment":"Legitimate companies use official email. WhatsApp job offers are almost always fraudulent.","author":"user_arjun","date":"2024-02-13","upvotes":55},
        {"comment_id":"C006","post_id":"P013","company_name":"Telegram Jobs","comment":"Telegram job channels with too-good offers always end up asking document fees or registration money.","author":"user_pooja","date":"2024-03-02","upvotes":41},
        {"comment_id":"C007","post_id":"P012","company_name":"ABC Career Services","comment":"ABC Career Services is same as ABC Placement Agency I think. Same phone number, different name.","author":"user_nisha","date":"2024-02-26","upvotes":27},
        {"comment_id":"C008","post_id":"P005","company_name":"Infosys","comment":"Infosys confirmed they never charge training fee. Their offer letters carry a verification QR code.","author":"user_kamal","date":"2024-02-02","upvotes":39},
    ]
    return pd.DataFrame(rows)

def gen_reports():
    rows = [
        {"report_id":"R001","company_name":"ABC Consulting","report_type":"Fake Internship","description":"Company collected Rs 2000 registration fee from 15 students for non-existent internship roles.","severity":"High","number_of_reports":8,"verified":True,"date":"2024-01-15","risk_score":78},
        {"report_id":"R002","company_name":"ABC Consultancy","report_type":"Security Deposit Fraud","description":"ABC Consultancy demanded Rs 5000 security deposit via UPI before releasing offer letter. Office untraceable.","severity":"High","number_of_reports":5,"verified":True,"date":"2024-01-18","risk_score":74},
        {"report_id":"R003","company_name":"TCS","report_type":"Brand Impersonation","description":"Scammers impersonating TCS HR via WhatsApp and personal Gmail IDs. Asking processing fee of Rs 5000.","severity":"Critical","number_of_reports":24,"verified":True,"date":"2024-01-20","risk_score":95},
        {"report_id":"R004","company_name":"Infosys","report_type":"Fake Offer Letter","description":"Forged Infosys offer letters distributed over Telegram. Candidates asked to pay training kit fee.","severity":"Critical","number_of_reports":19,"verified":True,"date":"2024-02-01","risk_score":91},
        {"report_id":"R005","company_name":"ABC Career Consulting","report_type":"Placement Scam","description":"Charged Rs 10000 for guaranteed MNC placement. No placement provided. Phones now switched off.","severity":"High","number_of_reports":11,"verified":True,"date":"2024-02-10","risk_score":85},
        {"report_id":"R006","company_name":"ABC Recruiters","report_type":"Joining Fee Fraud","description":"Called fee a joining contribution. Collected from 20 candidates. No joining letters issued after payment.","severity":"High","number_of_reports":9,"verified":True,"date":"2024-02-14","risk_score":80},
        {"report_id":"R007","company_name":"ABC Placement Agency","report_type":"Investment Scam","description":"Described fee as refundable investment. Rs 2000 collected per candidate. Refunds never processed.","severity":"Medium","number_of_reports":6,"verified":False,"date":"2024-02-17","risk_score":62},
        {"report_id":"R008","company_name":"Global Placement Hub","report_type":"Registration Scam","description":"Collected Rs 8000 each from 50+ students for MNC placement. Fake office address. Disappeared.","severity":"Critical","number_of_reports":31,"verified":True,"date":"2024-02-15","risk_score":97},
        {"report_id":"R009","company_name":"ABC Career Services","report_type":"Processing Fee Scam","description":"Demanded processing amount of Rs 1500 before connecting candidates to employers. No results delivered.","severity":"Medium","number_of_reports":4,"verified":False,"date":"2024-02-25","risk_score":55},
        {"report_id":"R010","company_name":"TCS iON","report_type":"Email Spoofing","description":"Using Gmail IDs mimicking TCS iON to send fake offer letters and collect document verification fee.","severity":"High","number_of_reports":13,"verified":True,"date":"2024-02-20","risk_score":83},
    ]
    return pd.DataFrame(rows)

def gen_verified_reports():
    rows = [
        {"verified_id":"V001","company_name":"ABC Consulting","category":"Fee Fraud","summary":"Multiple victims confirm ABC Consulting collected advance fees for fake internships.","total_victims":15,"total_amount_lost":30000,"date_verified":"2024-02-01","source":"CyberCell"},
        {"verified_id":"V002","company_name":"Global Placement Hub","category":"Placement Scam","summary":"Cybercrime registered FIR. Global Placement Hub collected fees from 50+ students.","total_victims":52,"total_amount_lost":416000,"date_verified":"2024-03-01","source":"CyberCell"},
        {"verified_id":"V003","company_name":"TCS","category":"Brand Impersonation","summary":"Verified impersonation network using WhatsApp to circulate forged TCS offer letters.","total_victims":38,"total_amount_lost":190000,"date_verified":"2024-02-15","source":"TCS Legal"},
        {"verified_id":"V004","company_name":"Infosys","category":"Offer Letter Forgery","summary":"Infosys legal team confirmed forged offer letters. Telegram channels used for distribution.","total_victims":29,"total_amount_lost":87000,"date_verified":"2024-02-20","source":"Infosys Legal"},
        {"verified_id":"V005","company_name":"ABC Career Consulting","category":"Placement Scam","summary":"ABC Career Consulting confirmed as scam entity. Registration cancelled.","total_victims":22,"total_amount_lost":220000,"date_verified":"2024-03-10","source":"DPIIT"},
    ]
    return pd.DataFrame(rows)


# ----------------------------------------------------------
#  Load or generate
# ----------------------------------------------------------

def load_or_generate(filename, generator, label):
    path = os.path.join(DATA_DIR, filename)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  📂 Loaded   {label:<35} → {len(df):>4} records")
    else:
        df = generator()
        df.to_csv(path, index=False)
        print(f"  🔧 Generated {label:<35} → {len(df):>4} records  [saved]")
    return df


print("\n📦 Loading Data Sources...")
print("-" * 65)
df_posts    = load_or_generate("community_posts.csv",    gen_posts,    "community_posts.csv")
df_comments = load_or_generate("community_comments.csv", gen_comments, "community_comments.csv")
df_reports  = load_or_generate("scam_reports.csv",       gen_reports,  "scam_reports.csv")
df_verified = load_or_generate("verified_scam_reports.csv", gen_verified_reports, "verified_scam_reports.csv")
print("-" * 65)
print(f"\n✅ Data Loading Complete.")
print(f"   Posts    : {len(df_posts)} | Comments : {len(df_comments)} | Reports : {len(df_reports)} | Verified : {len(df_verified)}")


📦 Loading Data Sources...
-----------------------------------------------------------------
  📂 Loaded   community_posts.csv                 →   12 records
  📂 Loaded   community_comments.csv              →   14 records
  📂 Loaded   scam_reports.csv                    →    8 records
  📂 Loaded   verified_scam_reports.csv           →    5 records
-----------------------------------------------------------------

✅ Data Loading Complete.
   Posts    : 12 | Comments : 14 | Reports : 8 | Verified : 5


---
## STEP 03 — Build Semantic Corpus

Combine all relevant text fields into a single `semantic_text` column
that the embedding model will encode into a dense vector.

```
semantic_text = company_name + ". " + title + ". " + content/description + ". " + tags
```

In [3]:
# ============================================================
#  STEP 03 — BUILD SEMANTIC CORPUS
# ============================================================

def safe_str(val):
    """Return a clean string, replacing NaN/None with empty string."""
    return "" if pd.isna(val) or val is None else str(val).strip()


def build_semantic_text(row, fields):
    """
    Concatenate specified fields into a single semantic text string.
    Sentence boundaries are preserved with '. ' separators.
    
    Parameters:
    -----------
    row    : pd.Series — a single DataFrame row
    fields : list[str] — column names to concatenate
    
    Returns: str
    """
    parts = [safe_str(row.get(f, "")) for f in fields]
    parts = [p for p in parts if p]  # drop empty parts
    return ". ".join(parts)


# ---------- Posts ----------
df_posts['semantic_text'] = df_posts.apply(
    build_semantic_text,
    fields=['company_name', 'title', 'content', 'tags'],
    axis=1
)
df_posts['source_type'] = 'post'
df_posts['record_id']   = df_posts['post_id']

# ---------- Comments ----------
df_comments['semantic_text'] = df_comments.apply(
    build_semantic_text,
    fields=['company_name', 'comment'],
    axis=1
)
df_comments['source_type'] = 'comment'
df_comments['record_id']   = df_comments['comment_id']

# ---------- Reports ----------
df_reports['semantic_text'] = df_reports.apply(
    build_semantic_text,
    fields=['company_name', 'report_type', 'description'],
    axis=1
)
df_reports['source_type'] = 'report'
df_reports['record_id']   = df_reports['report_id']

# ---------- Verified Reports ----------
df_verified['semantic_text'] = df_verified.apply(
    build_semantic_text,
    fields=['company_name', 'category', 'summary'],
    axis=1
)
df_verified['source_type'] = 'verified_report'
df_verified['record_id']   = df_verified['verified_id']

# ---------- Unified Corpus ----------
COMMON_COLS = ['record_id', 'source_type', 'company_name', 'semantic_text']

# Add any missing common cols
for df_ in [df_posts, df_comments, df_reports, df_verified]:
    for col in COMMON_COLS:
        if col not in df_.columns:
            df_[col] = ""

df_corpus = pd.concat(
    [
        df_posts[COMMON_COLS],
        df_comments[COMMON_COLS],
        df_reports[COMMON_COLS],
        df_verified[COMMON_COLS]
    ],
    ignore_index=True
).dropna(subset=['semantic_text'])

df_corpus = df_corpus[df_corpus['semantic_text'].str.strip() != ''].reset_index(drop=True)

print("📚 Semantic Corpus Built")
print("-" * 60)
print(f"  Posts          : {(df_corpus['source_type']=='post').sum()}")
print(f"  Comments       : {(df_corpus['source_type']=='comment').sum()}")
print(f"  Reports        : {(df_corpus['source_type']=='report').sum()}")
print(f"  Verified       : {(df_corpus['source_type']=='verified_report').sum()}")
print(f"  Total records  : {len(df_corpus)}")
print()
print("📋 Sample semantic_text entries:")
for i, row in df_corpus.head(4).iterrows():
    print(f"  [{row['source_type']:>15}] {row['semantic_text'][:90]}...")

📚 Semantic Corpus Built
------------------------------------------------------------
  Posts          : 12
  Comments       : 14
  Reports        : 8
  Verified       : 5
  Total records  : 39

📋 Sample semantic_text entries:
  [           post] ABC Consulting. Beware of ABC Consulting fake internship offer. They asked me to pay a reg...
  [           post] ABC Consultancy. ABC Consultancy demanded money before joining. ABC Consultancy Pvt Ltd se...
  [           post] TCS. Fake TCS offer letter circulating on WhatsApp. Someone is sending TCS offer letters v...
  [           post] TCS. TCS does not charge any fees — official clarification. TCS official website clearly s...


---
## STEP 04 — Load Embedding Model

Load `all-MiniLM-L6-v2` — a compact, fast, high-quality sentence embedding model.

| Property | Value |
|----------|-------|
| Model | `all-MiniLM-L6-v2` |
| Embedding dimensions | 384 |
| Max input tokens | 256 |
| Metric | Cosine similarity |

In [4]:
# ============================================================
#  STEP 04 — LOAD EMBEDDING MODEL
# ============================================================

MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Loading SentenceTransformer model: '{MODEL_NAME}'")
print("   (Downloads ~90 MB on first run — cached on subsequent runs)")
print()

embed_model = SentenceTransformer(MODEL_NAME)

# Quick sanity check — embed a test sentence
test_embedding = embed_model.encode(["This is a test sentence."])

print(f" Model loaded successfully.")
print(f"   Model name       : {MODEL_NAME}")
print(f"   Embedding dim    : {test_embedding.shape[1]}")
print(f"   Test vector norm : {np.linalg.norm(test_embedding[0]):.4f}  (expect ≈ 1.0 for normalized)")

Loading SentenceTransformer model: 'all-MiniLM-L6-v2'
   (Downloads ~90 MB on first run — cached on subsequent runs)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Model loaded successfully.
   Model name       : all-MiniLM-L6-v2
   Embedding dim    : 384
   Test vector norm : 1.0000  (expect ≈ 1.0 for normalized)


---
## STEP 05 — Generate Embeddings

Encode each `semantic_text` entry into a 384-dimensional vector.
Batch encoding is used for efficiency.

In [5]:
# ============================================================
#  STEP 05 — GENERATE EMBEDDINGS
# ============================================================

BATCH_SIZE = 32   # process 32 texts per forward pass

corpus_texts = df_corpus['semantic_text'].tolist()

print(f"  Generating embeddings for {len(corpus_texts)} corpus entries...")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   Expected dim: 384")
print()

corpus_embeddings = embed_model.encode(
    corpus_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True   # L2-normalise → cosine sim becomes dot product
)

# Attach embedding vectors back to the corpus DataFrame
df_corpus['embedding_vector'] = list(corpus_embeddings)

print(f"\n Embeddings generated.")
print(f"   Matrix shape : {corpus_embeddings.shape}  (records × dimensions)")
print(f"   dtype        : {corpus_embeddings.dtype}")
print(f"   Memory usage : {corpus_embeddings.nbytes / 1024:.1f} KB")

# Brief check: similarity between two clearly related sentences
sent_a = embed_model.encode(["registration fee scam"],       normalize_embeddings=True)
sent_b = embed_model.encode(["security deposit fraud"],      normalize_embeddings=True)
sent_c = embed_model.encode(["machine learning tutorial"],   normalize_embeddings=True)

sim_ab = cosine_similarity(sent_a, sent_b)[0][0]
sim_ac = cosine_similarity(sent_a, sent_c)[0][0]

print(f"\n Semantic Sanity Check:")
print(f"   'registration fee scam'  ↔  'security deposit fraud'   : {sim_ab:.4f}  (expect HIGH)")
print(f"   'registration fee scam'  ↔  'machine learning tutorial': {sim_ac:.4f}  (expect LOW)")

  Generating embeddings for 39 corpus entries...
   Batch size  : 32
   Expected dim: 384



Batches:   0%|          | 0/2 [00:00<?, ?it/s]


 Embeddings generated.
   Matrix shape : (39, 384)  (records × dimensions)
   dtype        : float32
   Memory usage : 58.5 KB

 Semantic Sanity Check:
   'registration fee scam'  ↔  'security deposit fraud'   : 0.4201  (expect HIGH)
   'registration fee scam'  ↔  'machine learning tutorial': 0.1304  (expect LOW)


---
## STEP 06 — Save Embeddings

Persist the embedding matrix and corpus metadata to disk using `joblib`.
This avoids re-encoding on every application restart.

In [6]:
# ============================================================
#  STEP 06 — SAVE EMBEDDINGS
# ============================================================

EMBEDDINGS_PATH = os.path.join(MODELS_DIR, "semantic_embeddings.pkl")

# ----------------------------------------------------------
#  Bundle everything needed to run searches later
# ----------------------------------------------------------
save_bundle = {
    "model_name":         MODEL_NAME,
    "embedding_dim":      corpus_embeddings.shape[1],
    "corpus_embeddings":  corpus_embeddings,           # numpy ndarray (N × 384)
    "corpus_texts":       corpus_texts,                # list of semantic_text strings
    "corpus_metadata":    df_corpus[['record_id', 'source_type', 'company_name']].to_dict(orient='records'),
    "generated_at":       datetime.now().isoformat(),
    "total_records":      len(corpus_texts)
}

joblib.dump(save_bundle, EMBEDDINGS_PATH, compress=3)

file_size_kb = os.path.getsize(EMBEDDINGS_PATH) / 1024

print(" Embeddings Saved")
print("-" * 55)
print(f"  Path           : {EMBEDDINGS_PATH}")
print(f"  File size      : {file_size_kb:.1f} KB")
print(f"  Records stored : {save_bundle['total_records']}")
print(f"  Embedding dim  : {save_bundle['embedding_dim']}")
print(f"  Saved at       : {save_bundle['generated_at']}")

# ----------------------------------------------------------
#  Verify reload
# ----------------------------------------------------------
loaded = joblib.load(EMBEDDINGS_PATH)
assert loaded['total_records'] == len(corpus_texts), "Reload mismatch!"
print(f"\n Reload verification passed — {loaded['total_records']} records intact.")

 Embeddings Saved
-------------------------------------------------------
  Path           : ../models/semantic_embeddings.pkl
  File size      : 57.5 KB
  Records stored : 39
  Embedding dim  : 384
  Saved at       : 2026-06-13T00:44:20.712859

 Reload verification passed — 39 records intact.


---
## STEP 07 — Semantic Search Function

Core search function:
1. Encode the user query with the same model
2. Compute cosine similarity against all corpus embeddings
3. Return top-K results above a similarity threshold

In [7]:
# ============================================================
#  STEP 07 — SEMANTIC SEARCH FUNCTION
# ============================================================

SIM_THRESHOLD = 0.35   # minimum cosine similarity to include a result
TOP_K_DEFAULT = 10     # default number of top results to return


def semantic_search(query,
                    top_k=TOP_K_DEFAULT,
                    threshold=SIM_THRESHOLD,
                    source_filter=None):
    """
    Search the corpus using semantic (vector) similarity.
    
    Parameters:
    -----------
    query         : str       — user search query (any natural language)
    top_k         : int       — maximum number of results to return
    threshold     : float     — minimum cosine similarity (0–1)
    source_filter : list[str] — optional filter: ['post','comment','report','verified_report']
    
    Returns:
    --------
    pd.DataFrame — top matching records with cosine_similarity score
    """
    if not query or not query.strip():
        return pd.DataFrame()
    
    # 1. Encode query
    query_vec = embed_model.encode(
        [query.strip()],
        normalize_embeddings=True
    )  # shape: (1, 384)
    
    # 2. Compute cosine similarity (dot product because both are L2-normalised)
    sims = cosine_similarity(query_vec, corpus_embeddings)[0]  # shape: (N,)
    
    # 3. Build results DataFrame
    results = df_corpus.copy()
    results['cosine_similarity'] = sims
    
    # 4. Filter by threshold
    results = results[results['cosine_similarity'] >= threshold]
    
    # 5. Optional source-type filter
    if source_filter:
        results = results[results['source_type'].isin(source_filter)]
    
    # 6. Rank and trim
    results = (
        results
        .sort_values('cosine_similarity', ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    
    return results


# ----------------------------------------------------------
#  Demo
# ----------------------------------------------------------

print(" Semantic Search — Demo")
print("=" * 65)

demo_query = "company asking for money before giving offer letter"
print(f"Query: \"{demo_query}\"")
print(f"Threshold: {SIM_THRESHOLD}  |  Top-K: {TOP_K_DEFAULT}\n")

demo_results = semantic_search(demo_query)

if demo_results.empty:
    print("  No results above threshold.")
else:
    print(demo_results[['record_id','source_type','company_name','cosine_similarity','semantic_text']]
          .assign(semantic_text=lambda df: df['semantic_text'].str[:70] + '...')
          .to_string(index=False))

print(f"\n Semantic search function ready. Found {len(demo_results)} results.")

 Semantic Search — Demo
Query: "company asking for money before giving offer letter"
Threshold: 0.35  |  Top-K: 10

record_id source_type    company_name  cosine_similarity                                                             semantic_text
     P002        post ABC Consultancy           0.495648 ABC Consultancy. ABC Consultancy demanded money before joining. ABC Co...
     R002      report ABC Consultancy           0.435500 ABC Consultancy. Security Deposit. ABC Consultancy demanded security d...
     C001     comment  ABC Consulting           0.350394 ABC Consulting. I also got the same message from ABC Consulting. Defin...

 Semantic search function ready. Found 3 results.


---
## STEP 08 — Similar Company Discovery

Find companies that are semantically similar to a given company name.
This handles name variations, rebranding, and synonym cases:
```
ABC Consulting → ABC Consultancy, ABC Career Services,
                 ABC Recruiters, ABC Placement Agency
```

In [8]:
# ============================================================
#  STEP 08 — SIMILAR COMPANY DISCOVERY
# ============================================================

def find_similar_companies(company_name, top_n=10, threshold=0.30):
    """
    Discover companies semantically similar to the given name.
    
    Strategy:
      1. Embed the company name
      2. Compare against ALL corpus embeddings
      3. Collect unique company names above the threshold
      4. Aggregate: max similarity + total mention count
    
    Parameters:
    -----------
    company_name : str   — query company name
    top_n        : int   — how many similar companies to return
    threshold    : float — minimum cosine similarity
    
    Returns:
    --------
    pd.DataFrame with columns:
        company_name, max_similarity, mention_count, sources
    """
    query_vec = embed_model.encode(
        [company_name.strip()],
        normalize_embeddings=True
    )
    
    sims = cosine_similarity(query_vec, corpus_embeddings)[0]
    
    results = df_corpus.copy()
    results['similarity'] = sims
    results = results[results['similarity'] >= threshold]
    
    if results.empty:
        return pd.DataFrame(columns=['company_name','max_similarity','mention_count','sources'])
    
    # Aggregate by company name
    agg = (
        results.groupby('company_name')
        .agg(
            max_similarity=('similarity',    'max'),
            mention_count =('record_id',     'count'),
            sources       =('source_type',   lambda x: ', '.join(sorted(x.unique())))
        )
        .reset_index()
        .sort_values('max_similarity', ascending=False)
    )
    
    # Exclude the exact query company from the "similar" list if desired
    query_lower = company_name.strip().lower()
    similar_only = agg[agg['company_name'].str.lower() != query_lower]
    
    return similar_only.head(top_n).reset_index(drop=True)


# ----------------------------------------------------------
#  Demo
# ----------------------------------------------------------

print(" Similar Company Discovery — Demo")
print("=" * 65)

for company in ["ABC Consulting", "TCS", "Infosys"]:
    similar = find_similar_companies(company)
    print(f"\n  Query: \"{company}\"  →  {len(similar)} similar companies found")
    if not similar.empty:
        print(similar[['company_name','max_similarity','mention_count','sources']]
              .to_string(index=False))

print("\n Similar company discovery ready.")

 Similar Company Discovery — Demo

  Query: "ABC Consulting"  →  4 similar companies found
          company_name  max_similarity  mention_count                                sources
 ABC Career Consulting        0.722651              4 comment, post, report, verified_report
ABC Consulting Pvt Ltd        0.642584              3                  comment, post, report
       ABC Consultancy        0.566378              3                  comment, post, report
  Global Placement Hub        0.348193              2                report, verified_report

  Query: "TCS"  →  1 similar companies found
company_name  max_similarity  mention_count               sources
     TCS iON        0.434147              3 comment, post, report

  Query: "Infosys"  →  1 similar companies found
company_name  max_similarity  mention_count sources
 Infosys BPM        0.477034              1    post

 Similar company discovery ready.


---
## STEP 09 — Scam Pattern Discovery

Find reports that describe **semantically similar scam patterns**,
even when completely different words are used.

```
'registration fee'  ≈  'security deposit'
                    ≈  'joining contribution'
                    ≈  'processing amount'
                    ≈  'investment required'
```

In [9]:
# ============================================================
#  STEP 09 — SCAM PATTERN DISCOVERY
# ============================================================

# Known scam pattern anchors — used to build a pattern cluster map
SCAM_PATTERN_ANCHORS = [
    "registration fee",
    "security deposit",
    "joining contribution",
    "processing amount",
    "investment required",
    "training kit fee",
    "document verification fee",
    "brand impersonation offer letter",
    "WhatsApp fake job offer",
    "Telegram job scam channel"
]


def find_similar_scam_patterns(text, top_k=8, threshold=0.30, source_filter=None):
    """
    Find corpus entries describing a semantically similar scam pattern.
    
    Parameters:
    -----------
    text          : str       — free-text description of a scam pattern
    top_k         : int       — max results
    threshold     : float     — minimum similarity
    source_filter : list[str] — restrict to specific source types
    
    Returns:
    --------
    pd.DataFrame — semantically similar records
    """
    return semantic_search(
        query=text,
        top_k=top_k,
        threshold=threshold,
        source_filter=source_filter
    )


def build_pattern_cluster_map(anchors, top_k=5, threshold=0.30):
    """
    Pre-compute a cluster map: each anchor → top semantically similar records.
    Useful for the Community Intelligence Layer (Step 10).
    
    Returns:
    --------
    dict  { anchor_text : [list of matching company names] }
    """
    cluster_map = {}
    for anchor in anchors:
        matches = find_similar_scam_patterns(anchor, top_k=top_k, threshold=threshold)
        if not matches.empty:
            companies = matches['company_name'].unique().tolist()
        else:
            companies = []
        cluster_map[anchor] = companies
    return cluster_map


# ----------------------------------------------------------
#  Demo
# ----------------------------------------------------------

print(" Scam Pattern Discovery — Demo")
print("=" * 65)

pattern_queries = [
    "registration fee",
    "security deposit",
    "joining contribution",
    "processing amount",
    "investment required"
]

for pq in pattern_queries:
    matches = find_similar_scam_patterns(pq, top_k=4)
    print(f"\n  Pattern: \"{pq}\"  →  {len(matches)} matches")
    if not matches.empty:
        for _, row in matches.iterrows():
            print(f"    [{row['source_type']:>15}] {row['cosine_similarity']:.3f}  {row['company_name']:<25}  {row['semantic_text'][:55]}...")

# Build pattern cluster map
print("\n  Building Pattern Cluster Map...")
pattern_cluster_map = build_pattern_cluster_map(SCAM_PATTERN_ANCHORS)

print("\n Pattern Cluster Map:")
for pattern, companies in pattern_cluster_map.items():
    print(f"  '{pattern}' → {companies}")

print("\n Scam pattern discovery ready.")

 Scam Pattern Discovery — Demo

  Pattern: "registration fee"  →  4 matches
    [        comment] 0.542  ABC Consulting             ABC Consulting. Registration fee is the #1 red flag. An...
    [           post] 0.450  TCS                        TCS. TCS does not charge any fees — official clarificat...
    [           post] 0.397  Global Placement Hub       Global Placement Hub. Global Placement Hub charged and ...
    [        comment] 0.395  Infosys                    Infosys. Infosys never asks for training fee. Their off...

  Pattern: "security deposit"  →  3 matches
    [        comment] 0.515  ABC Consultancy            ABC Consultancy. Same experience. ABC Consultancy asked...
    [         report] 0.500  ABC Consultancy            ABC Consultancy. Security Deposit. ABC Consultancy dema...
    [           post] 0.454  ABC Consultancy            ABC Consultancy. ABC Consultancy demanded money before ...

  Pattern: "joining contribution"  →  0 matches

  Pattern: "processing a

---
## STEP 10 — Community Intelligence Layer

Aggregate community signals to produce:
- **Most reported companies**
- **Similar scam clusters**
- **Trending scam patterns**
- **Frequent recruiter / brand names**

In [10]:
print(df_reports.columns.tolist())

['report_id', 'company_name', 'report_type', 'description', 'reported_by', 'date', 'severity', 'number_of_reports', 'verified', 'semantic_text', 'source_type', 'record_id']


In [11]:
df_reports.head(3)

,report_id,company_name,report_type,description,reported_by,date,severity,number_of_reports,verified,semantic_text,source_type,record_id
0,R001,ABC Consulting,Fake Internship,ABC Consulting collected Rs 2000 as registration fee for internship. Company...,user_rahul,2024-01-15,High,8,True,ABC Consulting. Fake Internship. ABC Consulting collected Rs 2000 as registr...,report,R001
1,R002,ABC Consultancy,Security Deposit,ABC Consultancy demanded security deposit of Rs 5000 before issuing joining ...,user_priya,2024-01-18,High,5,True,ABC Consultancy. Security Deposit. ABC Consultancy demanded security deposit...,report,R002
2,R003,TCS,Impersonation,Scammers impersonating TCS HR via WhatsApp and Gmail asking processing fees.,user_amit,2024-01-20,Critical,24,True,TCS. Impersonation. Scammers impersonating TCS HR via WhatsApp and Gmail ask...,report,R003


In [12]:
# ============================================================
#  STEP 10 — COMMUNITY INTELLIGENCE LAYER
# ============================================================

def build_community_intelligence():
    """
    Aggregate intelligence signals from all community data sources.
    
    Returns:
    --------
    dict containing:
      - most_reported_companies  : ranked list
      - scam_clusters            : pattern → companies map
      - trending_patterns        : top scam method keywords
      - frequent_company_names   : most mentioned names
      - total_verified_victims   : int
      - total_amount_lost        : int (INR)
    """
    
    # ── Most reported companies (from scam_reports) ──
    report_counts = (
        df_reports
        .groupby('company_name')
        .agg(
            total_reports    =('number_of_reports', 'sum'),
            report_entries   =('report_id', 'count'),
            
            verified_count   =('verified', 'sum'),
            top_severity     =('severity', lambda x: x.value_counts().index[0])
        )
        .reset_index()
        .sort_values('total_reports', ascending=False)
    )
    most_reported = report_counts.head(10).to_dict(orient='records')
    
    # ── Scam clusters from pattern map ──
    scam_clusters = {
        pattern: companies
        for pattern, companies in pattern_cluster_map.items()
        if companies  # only include non-empty
    }
    
    # ── Trending scam patterns (term frequency in report descriptions) ──
    fee_keywords = [
        'registration fee', 'security deposit', 'joining contribution',
        'processing amount', 'investment required', 'training fee',
        'document fee', 'verification fee', 'placement fee', 'advance payment'
    ]
    all_descriptions = ' '.join(df_reports['description'].fillna('').str.lower())
    trending = {}
    for kw in fee_keywords:
        count = all_descriptions.count(kw)
        if count > 0:
            trending[kw] = count
    trending = dict(sorted(trending.items(), key=lambda x: x[1], reverse=True))
    
    # ── Frequent company names across ALL sources ──
    all_company_names = (
        list(df_posts['company_name'].dropna()) +
        list(df_comments['company_name'].dropna()) +
        list(df_reports['company_name'].dropna())
    )
    freq_names = Counter(all_company_names).most_common(10)
    
    # ── Verified victims & losses ──
    total_victims = int(
        pd.to_numeric(df_verified.get('total_victims', pd.Series()), errors='coerce').sum()
    )
    total_loss = int(
        pd.to_numeric(df_verified.get('total_amount_lost', pd.Series()), errors='coerce').sum()
    )
    
    return {
        "most_reported_companies":  most_reported,
        "scam_clusters":            scam_clusters,
        "trending_patterns":        trending,
        "frequent_company_names":   dict(freq_names),
        "total_verified_victims":   total_victims,
        "total_amount_lost_inr":    total_loss,
        "generated_at":             datetime.now().isoformat()
    }


intelligence = build_community_intelligence()

print(" Community Intelligence Layer")
print("=" * 65)

print("\n Most Reported Companies:")
print(f"  {'Company':<30} {'Total Reports':>14} {'Avg Risk':>10} {'Severity':<12}")
print("  " + "-" * 68)
for r in intelligence['most_reported_companies']:
    print(f"  {r['company_name']:<30} {r['total_reports']:>14} {r['top_severity']:<12}")

print("\n Trending Scam Patterns (keyword frequency in reports):")
for pattern, count in intelligence['trending_patterns'].items():
    bar = '█' * (count * 3)
    print(f"  {pattern:<28} {bar} ({count})")

print("\n Most Mentioned Companies (all sources):")
for name, cnt in intelligence['frequent_company_names'].items():
    print(f"  {name:<30} : {cnt} mention(s)")

print(f"\n Total Verified Victims   : {intelligence['total_verified_victims']}")
print(f" Total Amount Lost (INR)  : ₹{intelligence['total_amount_lost_inr']:,}")
print("\n Community intelligence layer ready.")

 Community Intelligence Layer

 Most Reported Companies:
  Company                         Total Reports   Avg Risk Severity    
  --------------------------------------------------------------------
  Global Placement Hub                       31 Critical    
  TCS                                        24 Critical    
  Infosys                                    19 Critical    
  TCS iON                                    13 High        
  ABC Career Consulting                      11 High        
  ABC Consulting                              8 High        
  ABC Consulting Pvt Ltd                      6 High        
  ABC Consultancy                             5 High        

 Trending Scam Patterns (keyword frequency in reports):
  registration fee             ███ (1)
  security deposit             ███ (1)
  document fee                 ███ (1)

 Most Mentioned Companies (all sources):
  Infosys                        : 6 mention(s)
  ABC Consulting                 : 5 mention(s)


---
## STEP 11 — Risk Aggregation

Combine signals from three engines to produce a `community_risk_score` (0–100):

| Signal | Weight |
|--------|--------|
| Detection Engine risk_score | 40% |
| Community report count & severity | 35% |
| Semantic similarity to known scams | 25% |

In [13]:
# ============================================================
#  STEP 11 — RISK AGGREGATION
# ============================================================

SEVERITY_SCORE = {'critical': 100, 'high': 75, 'medium': 50, 'low': 25}


def compute_community_risk_score(company_name,
                                  detection_risk_score=None,
                                  sem_threshold=0.30):
    """
    Compute a composite community risk score (0–100) for a company.
    
    Components:
    -----------
    A) Detection Engine score  — passed in from Notebook 5 (or estimated)
    B) Community report score  — severity + report count + verified flag
    C) Semantic similarity score — how similar to known scam patterns
    
    Formula:
    --------
    community_risk_score = 0.40 * A + 0.35 * B + 0.25 * C
    
    Parameters:
    -----------
    company_name         : str   — company to assess
    detection_risk_score : float — score from Notebook 5 (0–100); None → estimated
    sem_threshold        : float — cosine sim threshold for semantic component
    
    Returns:
    --------
    dict — score breakdown and final score
    """
    
    # ── A: Detection Engine Score ──
    if detection_risk_score is not None:
        score_A = float(detection_risk_score)
    else:
        # Estimate from scam_reports if Notebook 5 score unavailable
        match = df_reports[df_reports['company_name'].str.lower() == company_name.lower()]
        score_A = float(match['risk_score'].mean()) if not match.empty else 50.0
    
    # ── B: Community Report Score ──
    match_reports = df_reports[
        df_reports['company_name'].str.lower().str.contains(
            company_name.lower()[:8], na=False
        )
    ]
    
    if match_reports.empty:
        score_B = 0.0
    else:
        sev_scores   = match_reports['severity'].str.lower().map(SEVERITY_SCORE).fillna(25)
        nr           = pd.to_numeric(match_reports['number_of_reports'], errors='coerce').fillna(1)
        verified_mul = match_reports['verified'].apply(lambda v: 1.2 if v else 1.0)
        
        raw_B = float((sev_scores * np.log1p(nr) * verified_mul).mean())
        score_B = min(100.0, raw_B)
    
    # ── C: Semantic Similarity Score ──
    sem_results = semantic_search(company_name, top_k=5, threshold=sem_threshold,
                                   source_filter=['report','verified_report'])
    if sem_results.empty:
        score_C = 0.0
    else:
        # Scale max cosine similarity (0–1) → risk contribution (0–100)
        score_C = float(sem_results['cosine_similarity'].max()) * 100
    
    # ── Final weighted score ──
    final_score = round(0.40 * score_A + 0.35 * score_B + 0.25 * score_C, 2)
    final_score = min(100.0, max(0.0, final_score))
    
    # ── Risk label ──
    if final_score >= 80:
        risk_label = "CRITICAL RISK"
    elif final_score >= 60:
        risk_label = "HIGH RISK"
    elif final_score >= 40:
        risk_label = "MODERATE RISK"
    elif final_score >= 20:
        risk_label = "LOW RISK"
    else:
        risk_label = "MINIMAL RISK"
    
    return {
        "company_name":           company_name,
        "detection_score_A":      round(score_A, 2),
        "community_report_score_B": round(score_B, 2),
        "semantic_sim_score_C":   round(score_C, 2),
        "community_risk_score":   final_score,
        "risk_label":             risk_label
    }


# ----------------------------------------------------------
#  Demo — Risk Aggregation
# ----------------------------------------------------------

print("⚡ Risk Aggregation — Demo")
print("=" * 65)

test_companies = [
    ("ABC Consulting",        78.0),
    ("TCS",                   95.0),
    ("Infosys",               91.0),
    ("Global Placement Hub",  97.0),
    ("ABC Career Services",   None),
]

print(f"\n  {'Company':<28} {'Score A':>8} {'Score B':>8} {'Score C':>8} {'FINAL':>8}  {'Risk Label'}")
print("  " + "-" * 85)

risk_results = {}
for company, det_score in test_companies:
    r = compute_community_risk_score(company, det_score)
    risk_results[company] = r
    print(f"  {company:<28} {r['detection_score_A']:>8.1f} "
          f"{r['community_report_score_B']:>8.1f} "
          f"{r['semantic_sim_score_C']:>8.1f} "
          f"{r['community_risk_score']:>8.1f}  {r['risk_label']}")

print("\n Risk aggregation ready.")

⚡ Risk Aggregation — Demo

  Company                       Score A  Score B  Score C    FINAL  Risk Label
  -------------------------------------------------------------------------------------
  ABC Consulting                   78.0    100.0     68.8     83.4  CRITICAL RISK
  TCS                              95.0    100.0     52.0     86.0  CRITICAL RISK
  Infosys                          91.0    100.0     57.3     85.7  CRITICAL RISK
  Global Placement Hub             97.0    100.0     61.8     89.2  CRITICAL RISK
  ABC Career Services              50.0    100.0     64.4     71.1  HIGH RISK

 Risk aggregation ready.


---
## STEP 12 — Frontend Search Response

Build a complete, structured JSON response suitable for the
Community Page frontend or downstream API consumers.

In [14]:
# ============================================================
#  STEP 12 — FRONTEND SEARCH RESPONSE
# ============================================================

def build_frontend_response(query,
                             detection_risk_score=None,
                             top_k=10,
                             sem_threshold=0.30):
    """
    Full pipeline → returns API-ready JSON response for the frontend.
    
    Parameters:
    -----------
    query                : str   — user's search query
    detection_risk_score : float — from Notebook 5 (optional)
    top_k                : int   — max results per category
    sem_threshold        : float — cosine similarity cutoff
    
    Returns:
    --------
    dict — structured API response
    """
    
    # 1. Semantic search — full corpus
    sem_results = semantic_search(query, top_k=top_k, threshold=sem_threshold)
    
    # 2. Similar companies
    similar_companies = find_similar_companies(query, top_n=10, threshold=sem_threshold)
    
    # 3. Related reports (from reports + verified only)
    related_reports = semantic_search(
        query, top_k=5, threshold=sem_threshold,
        source_filter=['report', 'verified_report']
    )
    
    # 4. Similar scam patterns
    scam_patterns = find_similar_scam_patterns(query, top_k=5, threshold=sem_threshold)
    
    # 5. Risk score
    risk = compute_community_risk_score(query, detection_risk_score, sem_threshold)
    
    # 6. Helper: DataFrame → list of records, limiting text length
    def df_to_list(df, cols, text_cols=None, text_limit=120):
        if df.empty:
            return []
        available = [c for c in cols if c in df.columns]
        out = df[available].fillna("").to_dict(orient='records')
        if text_cols:
            for rec in out:
                for tc in text_cols:
                    if tc in rec:
                        rec[tc] = str(rec[tc])[:text_limit] + ("..." if len(str(rec[tc])) > text_limit else "")
        # Round floats
        for rec in out:
            for k, v in rec.items():
                if isinstance(v, (float, np.floating)):
                    rec[k] = round(float(v), 4)
        return out
    
    response = {
        "query":                 query,
        "community_risk_score":  risk['community_risk_score'],
        "risk_label":            risk['risk_label'],
        "score_breakdown": {
            "detection_score":        risk['detection_score_A'],
            "community_report_score": risk['community_report_score_B'],
            "semantic_sim_score":     risk['semantic_sim_score_C']
        },
        "total_semantic_matches":  len(sem_results),
        "similar_companies":       df_to_list(
            similar_companies,
            ['company_name', 'max_similarity', 'mention_count', 'sources']
        ),
        "related_reports":         df_to_list(
            related_reports,
            ['record_id', 'source_type', 'company_name', 'semantic_text', 'cosine_similarity'],
            text_cols=['semantic_text']
        ),
        "similar_scam_patterns":   df_to_list(
            scam_patterns,
            ['record_id', 'source_type', 'company_name', 'semantic_text', 'cosine_similarity'],
            text_cols=['semantic_text']
        ),
        "top_results":             df_to_list(
            sem_results,
            ['record_id', 'source_type', 'company_name', 'semantic_text', 'cosine_similarity'],
            text_cols=['semantic_text']
        ),
        "generated_at":            datetime.now().isoformat()
    }
    
    return response


# ----------------------------------------------------------
#  Demo
# ----------------------------------------------------------

print(" Frontend Search Response — Demo")
print("=" * 65)

fr = build_frontend_response("ABC Consulting", detection_risk_score=78.0)

# Print compact summary (not full nested results)
compact_keys = [k for k in fr if k not in ('top_results','related_reports','similar_scam_patterns','top_results')]
print(json.dumps({k: fr[k] for k in compact_keys}, indent=2))

print(f"\n  similar_companies    → {len(fr['similar_companies'])} companies")
print(f"  related_reports      → {len(fr['related_reports'])} records")
print(f"  similar_scam_patterns→ {len(fr['similar_scam_patterns'])} records")
print(f"  top_results          → {len(fr['top_results'])} records")
print("\n Frontend response builder ready.")

 Frontend Search Response — Demo
{
  "query": "ABC Consulting",
  "community_risk_score": 83.4,
  "risk_label": "CRITICAL RISK",
  "score_breakdown": {
    "detection_score": 78.0,
    "community_report_score": 100.0,
    "semantic_sim_score": 68.8
  },
  "total_semantic_matches": 10,
  "similar_companies": [
    {
      "company_name": "ABC Career Consulting",
      "max_similarity": 0.7227,
      "mention_count": 4,
      "sources": "comment, post, report, verified_report"
    },
    {
      "company_name": "ABC Consulting Pvt Ltd",
      "max_similarity": 0.6426,
      "mention_count": 3,
      "sources": "comment, post, report"
    },
    {
      "company_name": "ABC Consultancy",
      "max_similarity": 0.5664,
      "mention_count": 3,
      "sources": "comment, post, report"
    },
    {
      "company_name": "Global Placement Hub",
      "max_similarity": 0.3482,
      "mention_count": 2,
      "sources": "report, verified_report"
    }
  ],
  "generated_at": "2026-06-13T00:44:

---
## STEP 13 — Test Cases

Run the complete pipeline against six representative queries.

In [15]:
# ============================================================
#  STEP 13 — TEST CASES
# ============================================================

TEST_CASES = [
    {"query": "ABC Consulting",   "detection_score": 78.0},
    {"query": "TCS",              "detection_score": 95.0},
    {"query": "Infosys",          "detection_score": 91.0},
    {"query": "Registration Fee", "detection_score": None},
    {"query": "WhatsApp",         "detection_score": None},
    {"query": "Telegram",         "detection_score": None},
]

SEP = "=" * 70
all_responses = {}

for i, tc in enumerate(TEST_CASES, 1):
    print(SEP)
    print(f"  TEST CASE {i}/{len(TEST_CASES)} : \"{tc['query']}\"")
    print(SEP)
    
    response = build_frontend_response(
        tc['query'],
        detection_risk_score=tc['detection_score']
    )
    all_responses[tc['query']] = response
    
    # ── Core Metrics ──
    print(f"  {'Query':<30}: {response['query']}")
    print(f"  {'Community Risk Score':<30}: {response['community_risk_score']} / 100")
    print(f"  {'Risk Label':<30}: {response['risk_label']}")
    print(f"  {'Score Breakdown':<30}: Detection={response['score_breakdown']['detection_score']}  "
          f"Community={response['score_breakdown']['community_report_score']}  "
          f"Semantic={response['score_breakdown']['semantic_sim_score']}")
    print(f"  {'Total Semantic Matches':<30}: {response['total_semantic_matches']}")
    
    # ── Similar Companies ──
    if response['similar_companies']:
        print(f"\n   Similar Companies ({len(response['similar_companies'])})")
        for c in response['similar_companies'][:5]:
            print(f"     Sim={c['max_similarity']:.3f}  {c['company_name']:<30}  ({c['mention_count']} mentions)")
    
    # ── Related Reports ──
    if response['related_reports']:
        print(f"\n   Related Reports ({len(response['related_reports'])})")
        for r in response['related_reports'][:3]:
            print(f"     [{r.get('record_id','?'):>5}] Sim={r['cosine_similarity']:.3f}  {r['company_name']:<22}  {r['semantic_text'][:55]}...")
    
    # ── Scam Patterns ──
    if response['similar_scam_patterns']:
        print(f"\n   Similar Scam Patterns ({len(response['similar_scam_patterns'])})")
        for sp in response['similar_scam_patterns'][:3]:
            print(f"     [{sp.get('record_id','?'):>5}] Sim={sp['cosine_similarity']:.3f}  {sp['company_name']:<22}  {sp['semantic_text'][:55]}...")
    
    print()

print(SEP)
print(" All test cases completed.")
print(SEP)

  TEST CASE 1/6 : "ABC Consulting"
  Query                         : ABC Consulting
  Community Risk Score          : 83.4 / 100
  Risk Label                    : CRITICAL RISK
  Score Breakdown               : Detection=78.0  Community=100.0  Semantic=68.8
  Total Semantic Matches        : 10

   Similar Companies (4)
     Sim=0.723  ABC Career Consulting           (4 mentions)
     Sim=0.643  ABC Consulting Pvt Ltd          (3 mentions)
     Sim=0.566  ABC Consultancy                 (3 mentions)
     Sim=0.348  Global Placement Hub            (2 mentions)

   Related Reports (5)
     [ R005] Sim=0.688  ABC Career Consulting   ABC Career Consulting. Placement Scam. Charged Rs 10000...
     [ V005] Sim=0.672  ABC Career Consulting   ABC Career Consulting. Placement Scam. ABC Career Consu...
     [ R007] Sim=0.637  ABC Consulting Pvt Ltd  ABC Consulting Pvt Ltd. Fake Internship. Same operators...

   Similar Scam Patterns (5)
     [ C013] Sim=0.734  ABC Consulting          ABC Consulti

In [16]:
# ----------------------------------------------------------
#  Test Cases — Summary Table
# ----------------------------------------------------------

print("\n Test Case Summary Table")
print("-" * 100)
print(f"{'Query':<22} {'Risk Score':>11} {'Similar Co.':>12} {'Reports':>8} {'Scam Patt.':>11}  {'Risk Label'}")
print("-" * 100)

for query, resp in all_responses.items():
    print(
        f"{query:<22}"
        f"{resp['community_risk_score']:>11.1f}"
        f"{len(resp['similar_companies']):>12}"
        f"{len(resp['related_reports']):>8}"
        f"{len(resp['similar_scam_patterns']):>11}"
        f"  {resp['risk_label']}"
    )

print("-" * 100)
print("\n Summary table complete.")


 Test Case Summary Table
----------------------------------------------------------------------------------------------------
Query                   Risk Score  Similar Co.  Reports  Scam Patt.  Risk Label
----------------------------------------------------------------------------------------------------
ABC Consulting               83.4           4       5          5  CRITICAL RISK
TCS                          86.0           1       3          5  CRITICAL RISK
Infosys                      85.7           1       2          5  CRITICAL RISK
Registration Fee             29.3           6       2          5  LOW RISK
WhatsApp                     33.0           3       3          5  LOW RISK
Telegram                     35.4           3       5          5  LOW RISK
----------------------------------------------------------------------------------------------------

 Summary table complete.


---
## STEP 14 — Export Results

Persist:
1. `../models/semantic_embeddings.pkl` — embedding matrix + metadata
2. `../data/semantic_search_index.csv` — full corpus with embeddings as text references

In [17]:
# ============================================================
#  STEP 14 — EXPORT RESULTS
# ============================================================

# ----------------------------------------------------------
#  14-A : Save Embeddings (full rebuild — latest state)
# ----------------------------------------------------------

EMBEDDINGS_OUT = os.path.join(MODELS_DIR, "semantic_embeddings.pkl")

final_bundle = {
    "model_name":           MODEL_NAME,
    "embedding_dim":        int(corpus_embeddings.shape[1]),
    "total_records":        len(corpus_texts),
    "corpus_embeddings":    corpus_embeddings,
    "corpus_texts":         corpus_texts,
    "corpus_metadata":      df_corpus[['record_id', 'source_type', 'company_name']].to_dict(orient='records'),
    "pattern_cluster_map":  pattern_cluster_map,
    "community_intelligence": {
        k: v for k, v in intelligence.items()
        if k not in ('scam_clusters',)   # skip non-serialisable nested dfs
    },
    "generated_at":         datetime.now().isoformat()
}

joblib.dump(final_bundle, EMBEDDINGS_OUT, compress=3)
emb_size_kb = os.path.getsize(EMBEDDINGS_OUT) / 1024

print(" Export 1 — Semantic Embeddings")
print("-" * 55)
print(f"  Path         : {EMBEDDINGS_OUT}")
print(f"  File size    : {emb_size_kb:.1f} KB")
print(f"  Records      : {final_bundle['total_records']}")
print(f"  Dimensions   : {final_bundle['embedding_dim']}")

# ----------------------------------------------------------
#  14-B : Save Semantic Search Index CSV
# ----------------------------------------------------------

INDEX_OUT = os.path.join(DATA_DIR, "semantic_search_index.csv")

# Build index — reference embedding by row index (not storing 384-dim vector in CSV)
index_df = df_corpus[['record_id', 'source_type', 'company_name', 'semantic_text']].copy()
index_df['embedding_index'] = index_df.index   # row position in corpus_embeddings matrix
index_df['vector_norm']     = [round(float(np.linalg.norm(v)), 6) for v in corpus_embeddings]

# Attach engagement / risk metadata from original sources
post_meta     = df_posts[['post_id','likes','number_of_comments']].rename(columns={'post_id':'record_id'})
report_meta   = df_reports[['report_id','number_of_reports','risk_score','severity','verified']].rename(columns={'report_id':'record_id'})
verified_meta = df_verified[['verified_id','total_victims','total_amount_lost']].rename(columns={'verified_id':'record_id'})

index_df = (
    index_df
    .merge(post_meta,     on='record_id', how='left')
    .merge(report_meta,   on='record_id', how='left')
    .merge(verified_meta, on='record_id', how='left')
)

index_df.to_csv(INDEX_OUT, index=False)
idx_size_kb = os.path.getsize(INDEX_OUT) / 1024

print("\n Export 2 — Semantic Search Index CSV")
print("-" * 55)
print(f"  Path         : {INDEX_OUT}")
print(f"  File size    : {idx_size_kb:.1f} KB")
print(f"  Total rows   : {len(index_df)}")
print(f"  Columns      : {list(index_df.columns)}")

print("\n Index Preview (top 5):")
print(index_df[['record_id','source_type','company_name','embedding_index','vector_norm']]
      .head(5).to_string(index=False))

print("\n" + "=" * 55)
print(" All exports complete.")
print(f"   {EMBEDDINGS_OUT}")
print(f"   {INDEX_OUT}")
print("=" * 55)

 Export 1 — Semantic Embeddings
-------------------------------------------------------
  Path         : ../models/semantic_embeddings.pkl
  File size    : 58.0 KB
  Records      : 39
  Dimensions   : 384


KeyError: "['risk_score'] not in index"

---

## 📋 Notebook Summary

| Step | Description | Status |
|------|-------------|--------|
| 01 | Import Libraries | ✅ |
| 02 | Load 4 data sources (posts / comments / reports / verified) | ✅ |
| 03 | Build semantic corpus (`semantic_text`) | ✅ |
| 04 | Load `all-MiniLM-L6-v2` embedding model | ✅ |
| 05 | Generate 384-dim embeddings, normalised | ✅ |
| 06 | Save embeddings to `../models/semantic_embeddings.pkl` | ✅ |
| 07 | `semantic_search(query)` — cosine similarity search | ✅ |
| 08 | `find_similar_companies(name)` — top-10 semantic variants | ✅ |
| 09 | `find_similar_scam_patterns(text)` — pattern cluster map | ✅ |
| 10 | Community Intelligence Layer — trends, clusters, losses | ✅ |
| 11 | `compute_community_risk_score()` — 3-signal aggregation | ✅ |
| 12 | `build_frontend_response()` — structured JSON API response | ✅ |
| 13 | Test cases — 6 queries with full output display | ✅ |
| 14 | Export `semantic_embeddings.pkl` + `semantic_search_index.csv` | ✅ |

---

### 🔗 Integration Points

| Component | This Notebook's Role |
|-----------|---------------------|
| Notebook 5 — Detection Engine | Provides `detection_risk_score` for Step 11 |
| Notebook 6 — URL Analysis | Report descriptions enriched before embedding |
| Notebook 7 — Community Search | Keyword pre-filter → Notebook 8 semantic re-rank |
| Frontend Community Search Bar | Calls `build_frontend_response(query)` |

---

### 📦 Exported Artefacts

```
../models/semantic_embeddings.pkl    ← embedding matrix + corpus metadata + cluster map
../data/semantic_search_index.csv    ← flat search index with embedding row references
```

---

### 🧪 Why `all-MiniLM-L6-v2`?

| Property | Value |
|----------|-------|
| Embedding size | 384 dimensions |
| Inference speed | ~14,000 sentences/sec (CPU) |
| Model size | ~90 MB |
| Semantic quality | State-of-the-art for sentence similarity |
| Licence | Apache 2.0 — commercial use permitted |

---

*Notebook 8 — Semantic Intelligence Engine | Fake Internship & Job Scam Detection System*